In [ ]:
import wrds
import pandas as pd
from fredapi import Fred
import numpy as np
from datetime import datetime, timedelta

# ============================================
# CONFIGURATION
# ============================================
START_DATE = '2015-01-01'
END_DATE = '2024-12-31'
FRED_API_KEY = '8e35578b7a38b7a6e296ce56a90fcce3'

# ============================================
# HELPER FUNCTIONS
# ============================================
def connect_with_retry(max_retries=3):
    """Connect to WRDS with retry logic"""
    for attempt in range(max_retries):
        try:
            print(f"Connection attempt {attempt + 1}/{max_retries}...")
            db = wrds.Connection()
            print("✓ Connected successfully")
            return db
        except Exception as e:
            print(f"✗ Connection failed: {e}")
            if attempt < max_retries - 1:
                print("Retrying in 5 seconds...")
                import time
                time.sleep(5)
            else:
                raise

def query_with_retry(db_conn, query, description, max_retries=3):
    """Execute query with retry logic for SSL timeouts"""
    db = db_conn
    for attempt in range(max_retries):
        try:
            print(f"  Attempt {attempt + 1}/{max_retries}: {description}")
            result = db.raw_sql(query)
            print(f"  ✓ Success: {len(result)} rows")
            return result
        except Exception as e:
            print(f"  ✗ Failed: {e}")
            if "SSL" in str(e) or "connection" in str(e).lower():
                if attempt < max_retries - 1:
                    print("  Reconnecting...")
                    try:
                        db.close()
                    except:
                        pass
                    db = connect_with_retry()
                else:
                    raise
            else:
                raise
    return pd.DataFrame()

# ============================================
# 1. CONNECT TO WRDS
# ============================================
print("Connecting to WRDS...")
db = connect_with_retry()

# ============================================
# 2. CREATE DATA DIRECTORY
# ============================================
import os
os.makedirs('data', exist_ok=True)
print("\nData directory created/verified")

# ============================================
# 3. BOND CHARACTERISTICS (Features 1-6) - FISD
# ============================================
# Check if already downloaded
if os.path.exists('data/bond_characteristics_raw.csv'):
    print("\nLoading existing bond characteristics from file...")
    bond_chars = pd.read_csv('data/bond_characteristics_raw.csv')
    print(f"Loaded {len(bond_chars)} bonds")
else:
    print("\nFetching bond characteristics from FISD...")
    print("Note: Querying year-by-year to avoid timeouts...")

    bond_chars_list = []
    years = range(2015, 2025)

    for year in years:
        year_start = f"{year}-01-01"
        year_end = f"{year}-12-31"

        query = f"""
            SELECT
                complete_cusip as cusip,
                issuer_cusip,
                maturity,
                coupon,
                offering_amt,
                offering_date,
                putable,
                redeemable,
                security_level as seniority_rank
            FROM fisd.fisd_mergedissue
            WHERE maturity >= '{START_DATE}'
                AND offering_date >= '{year_start}'
                AND offering_date <= '{year_end}'
                AND security_level = 'SEN'
        """

        result = query_with_retry(db, query, f"Bonds offered in {year}")
        if not result.empty:
            bond_chars_list.append(result)

    bond_chars = pd.concat(bond_chars_list, ignore_index=True) if bond_chars_list else pd.DataFrame()

    print(f"Found {len(bond_chars)} bonds")

    # Save immediately (raw data only)
    print("Saving bond_characteristics_raw.csv...")
    bond_chars.to_csv('data/bond_characteristics_raw.csv', index=False)

# ============================================
# 4. CREDIT RATINGS (Features 7-9) - FISD
# ============================================
if os.path.exists('data/credit_ratings_raw.csv'):
    print("\nLoading existing credit ratings from file...")
    ratings = pd.read_csv('data/credit_ratings_raw.csv')
    print(f"Loaded {len(ratings)} ratings")
else:
    print("\nFetching credit ratings from FISD...")
    print("Note: Querying year-by-year to avoid timeouts...")

    ratings_list = []
    years = range(2015, 2025)
    for year in years:
        year_start = f"{year}-01-01"
        year_end = f"{year}-12-31"

        query = f"""
            SELECT
                m.complete_cusip as cusip,
                r.rating_date,
                r.rating,
                r.rating_type
            FROM fisd.fisd_ratings r
            JOIN fisd.fisd_mergedissue m ON r.issue_id = m.issue_id
            WHERE r.rating_type IN ('SPR', 'MR', 'FR')
                AND r.rating_date >= '{year_start}'
                AND r.rating_date <= '{year_end}'
            ORDER BY m.complete_cusip, r.rating_date
        """

        result = query_with_retry(db, query, f"Ratings for {year}")
        if not result.empty:
            ratings_list.append(result)

    ratings = pd.concat(ratings_list, ignore_index=True) if ratings_list else pd.DataFrame()

    print(f"Found {len(ratings)} ratings")

    # Save immediately (raw data only)
    print("Saving credit_ratings_raw.csv...")
    ratings.to_csv('data/credit_ratings_raw.csv', index=False)

# ============================================
# 5. COMPUSTAT FUNDAMENTALS (Features 10-11, 33)
# ============================================
if os.path.exists('data/compustat_fundamentals_raw.csv'):
    print("\nLoading existing fundamentals from file...")
    fundamentals = pd.read_csv('data/compustat_fundamentals_raw.csv')
    print(f"Loaded {len(fundamentals)} company-quarters")
else:
    print("\nFetching fundamentals from Compustat...")
    print("Note: Querying year-by-year to avoid timeouts...")

    fundamentals_list = []
    years = range(2015, 2025)
    for year in years:
        year_start = f"{year}-01-01"
        year_end = f"{year}-12-31"

        query = f"""
            SELECT
                gvkey,
                datadate,
                cusip,
                sich as sic_code,
                dltt,  -- Long-term debt
                dlc,   -- Current debt
                ebitda,
                xint,  -- Interest expense
                ebit,
                at     -- Total assets
            FROM comp.funda
            WHERE datadate >= '{year_start}'
                AND datadate <= '{year_end}'
                AND indfmt = 'INDL'
                AND datafmt = 'STD'
                AND consol = 'C'
                AND ebitda IS NOT NULL
        """

        result = query_with_retry(db, query, f"Fundamentals for {year}")
        if not result.empty:
            fundamentals_list.append(result)

    fundamentals = pd.concat(fundamentals_list, ignore_index=True) if fundamentals_list else pd.DataFrame()

    print(f"Found {len(fundamentals)} company-quarters")

    # Save immediately
    print("Saving compustat_fundamentals_raw.csv...")
    fundamentals.to_csv('data/compustat_fundamentals_raw.csv', index=False)

# ============================================
# 6. TRACE TRADING DATA (Features 13-20)
# ============================================
if os.path.exists('data/trace_trading_raw.csv'):
    print("\nLoading existing trading data from file...")
    trading = pd.read_csv('data/trace_trading_raw.csv')
    print(f"Loaded {len(trading)} bond-months")
else:
    print("\nFetching trading data from WRDS Bond Returns...")
    print("Note: Querying year-by-year to avoid timeouts...")

    trading_list = []
    years = range(2015, 2025)
    for year in years:
        year_start = f"{year}-01-01"
        year_end = f"{year}-12-31"

        query = f"""
            SELECT
                cusip,
                date,
                t_volume,
                t_dvolume,
                t_spread,
                t_yld_pt,
                price_eom,
                ret_eom,
                rating_num
            FROM wrdsapps.bondret
            WHERE date >= '{year_start}'
                AND date <= '{year_end}'
        """

        result = query_with_retry(db, query, f"Trading data for {year}")
        if not result.empty:
            trading_list.append(result)

    trading = pd.concat(trading_list, ignore_index=True) if trading_list else pd.DataFrame()

    print(f"Found {len(trading)} bond-months")

    # Save immediately
    print("Saving trace_trading_raw.csv...")
    trading.to_csv('data/trace_trading_raw.csv', index=False)

# ============================================
# 7. TRACE MONTHLY EXTRA (Additional Features)
# ============================================
if os.path.exists('data/trace_monthly_extra_raw.csv'):
    print("\nLoading existing TRACE monthly extra data from file...")
    trace_monthly_extra = pd.read_csv('data/trace_monthly_extra_raw.csv')
    print(f"Loaded {len(trace_monthly_extra)} bond-months")
else:
    print("\nFetching additional TRACE features (aggregated monthly from raw data)...")
    print("Note: Querying in batches by bond and year, aggregating to monthly...")

    # Only get data for bonds we found
    all_cusips = bond_chars['cusip'].tolist() if not bond_chars.empty else []
    trace_monthly_extra_list = []

    if len(all_cusips) > 0:
        # Process in batches of 100 bonds at a time (can be larger since we're aggregating)
        batch_size = 100
        total_batches = (len(all_cusips) + batch_size - 1) // batch_size
        years = range(2015, 2025)

        for batch_idx in range(total_batches):
            start_idx = batch_idx * batch_size
            end_idx = min(start_idx + batch_size, len(all_cusips))
            batch_cusips = all_cusips[start_idx:end_idx]

            # For each batch of bonds, query year by year
            for year in years:
                year_start = f"{year}-01-01"
                year_end = f"{year}-12-31"

                cusip_list = "','".join(batch_cusips)
                query = f"""
                    SELECT
                        cusip_id,
                        DATE_TRUNC('month', trd_exctn_dt) as month,
                        COUNT(*) as trade_count,
                        SUM(entrd_vol_qt) as total_volume,
                        AVG(rptd_pr) as avg_price,
                        STDDEV(rptd_pr) as price_volatility,
                        AVG(yld_pt) as avg_yield,
                        COUNT(DISTINCT CASE WHEN rptg_party_type = 'D' THEN rptg_party_type END) as dealer_trades,
                        COUNT(DISTINCT CASE WHEN rptg_party_type = 'C' THEN rptg_party_type END) as customer_trades,
                        AVG(entrd_vol_qt) as avg_trade_size,
                        MAX(entrd_vol_qt) as max_trade_size,
                        MIN(entrd_vol_qt) as min_trade_size
                    FROM trace_enhanced.trace_enhanced
                    WHERE cusip_id IN ('{cusip_list}')
                        AND trd_exctn_dt >= '{year_start}'
                        AND trd_exctn_dt <= '{year_end}'
                        AND trc_st = 'T'
                    GROUP BY cusip_id, DATE_TRUNC('month', trd_exctn_dt)
                    ORDER BY cusip_id, month
                """

                result = query_with_retry(db, query,
                    f"TRACE monthly agg: Batch {batch_idx+1}/{total_batches}, Year {year}")
                if not result.empty:
                    trace_monthly_extra_list.append(result)

        trace_monthly_extra = pd.concat(trace_monthly_extra_list, ignore_index=True) if trace_monthly_extra_list else pd.DataFrame()
        print(f"Found {len(trace_monthly_extra)} bond-months from raw TRACE")
    else:
        print("No bonds to query for TRACE data")
        trace_monthly_extra = pd.DataFrame()

    # Save immediately
    if not trace_monthly_extra.empty:
        print("Saving trace_monthly_extra_raw.csv...")
        trace_monthly_extra.to_csv('data/trace_monthly_extra_raw.csv', index=False)

# ============================================
# 8. CRSP STOCK RETURNS (Feature 26)
# ============================================
if os.path.exists('data/crsp_returns_raw.csv'):
    print("\nLoading existing stock returns from file...")
    stock_returns = pd.read_csv('data/crsp_returns_raw.csv')
    print(f"Loaded {len(stock_returns)} stock-months")
else:
    print("\nFetching stock returns from CRSP...")

    # First get the CRSP-Compustat link (this is small, no need to chunk)
    ccmlink = query_with_retry(db, """
        SELECT gvkey, lpermno as permno, linkdt, linkenddt
        FROM crsp.ccmxpf_lnkhist
        WHERE linktype IN ('LU', 'LC')
            AND linkprim IN ('P', 'C')
    """, "CRSP-Compustat link table")

    # Then get returns year-by-year
    print("Note: Querying year-by-year to avoid timeouts...")
    stock_returns_list = []
    years = range(2015, 2025)
    for year in years:
        year_start = f"{year}-01-01"
        year_end = f"{year}-12-31"

        query = f"""
            SELECT
                permno,
                date,
                ret
            FROM crsp.msf
            WHERE date >= '{year_start}'
                AND date <= '{year_end}'
        """

        result = query_with_retry(db, query, f"Stock returns for {year}")
        if not result.empty:
            stock_returns_list.append(result)

    stock_returns = pd.concat(stock_returns_list, ignore_index=True) if stock_returns_list else pd.DataFrame()

    print(f"Found {len(stock_returns)} stock-months")

    # Save immediately
    print("Saving crsp_returns_raw.csv...")
    stock_returns.to_csv('data/crsp_returns_raw.csv', index=False)

# ============================================
# 9. MARKIT CDS SPREADS (Feature 27)
# ============================================
print("\nSkipping CDS data - not available in this WRDS subscription")
cds = pd.DataFrame()

# ============================================
# 10. FRED MACRO DATA (Features 21-23, 28)
# ============================================
if os.path.exists('data/fred_macro_raw.csv'):
    print("\nLoading existing macro data from file...")
    macro_data = pd.read_csv('data/fred_macro_raw.csv')
    print(f"Loaded macro data: {macro_data.shape}")
else:
    print("\nFetching macro data from FRED...")
    fred = Fred(api_key=FRED_API_KEY)

    # Download all macro series
    macro_series = {
        'VIX': 'VIXCLS',
        'HY_OAS': 'BAMLH0A0HYM2',
        'DGS10': 'DGS10',
        'DGS2': 'DGS2',
        'INFLATION_BREAKEVEN': 'T10YIE'
    }

    macro_data = pd.DataFrame()
    for name, series_id in macro_series.items():
        print(f"  Downloading {name}...")
        try:
            data = fred.get_series(series_id, observation_start=START_DATE, observation_end=END_DATE)
            if macro_data.empty:
                macro_data = pd.DataFrame(data, columns=[name])
            else:
                macro_data[name] = data
        except Exception as e:
            print(f"  Error downloading {name}: {e}")

    macro_data.reset_index(inplace=True)
    macro_data.rename(columns={'index': 'date'}, inplace=True)

    print(f"Downloaded macro data: {macro_data.shape}")

    # Save immediately
    print("Saving fred_macro_raw.csv...")
    macro_data.to_csv('data/fred_macro_raw.csv', index=False)

# ============================================
# 11. CLOSE WRDS CONNECTION
# ============================================
db.close()
print("\nWRDS connection closed.")

# ============================================
# 12. CALCULATE DERIVED FEATURES FOR FINAL DATASETS
# ============================================
print("\nCalculating derived features for final datasets...")

# Bond characteristics - calculate time-dependent features
if not bond_chars.empty:
    bond_chars['offering_date'] = pd.to_datetime(bond_chars['offering_date'])
    bond_chars['maturity'] = pd.to_datetime(bond_chars['maturity'])
    bond_chars['log_issue_size'] = np.log(bond_chars['offering_amt'])
    bond_chars['current_date'] = pd.Timestamp(END_DATE)
    bond_chars['time_to_maturity'] = (bond_chars['maturity'] - bond_chars['current_date']).dt.days / 365.25
    bond_chars['bond_age'] = (bond_chars['current_date'] - bond_chars['offering_date']).dt.days / 365.25

# Credit ratings - calculate derived features
if not ratings.empty:
    def rating_to_numeric(rating):
        """Convert letter rating to numeric scale 1-22"""
        rating_map = {
            'AAA': 1, 'AA+': 2, 'AA': 3, 'AA-': 4,
            'A+': 5, 'A': 6, 'A-': 7,
            'BBB+': 8, 'BBB': 9, 'BBB-': 10,
            'BB+': 11, 'BB': 12, 'BB-': 13,
            'B+': 14, 'B': 15, 'B-': 16,
            'CCC+': 17, 'CCC': 18, 'CCC-': 19,
            'CC': 20, 'C': 21, 'D': 22
        }
        return rating_map.get(rating, np.nan)

    ratings['numerical_rating'] = ratings['rating'].apply(rating_to_numeric)
    ratings['ig_indicator'] = (ratings['numerical_rating'] <= 10).astype(int)

# Fundamentals - calculate derived features
if not fundamentals.empty:
    fundamentals['total_debt'] = fundamentals['dltt'].fillna(0) + fundamentals['dlc'].fillna(0)
    fundamentals['leverage'] = fundamentals['total_debt'] / fundamentals['ebitda']
    fundamentals['interest_coverage'] = fundamentals['ebit'] / fundamentals['xint']

    # YoY profit growth (Feature 33)
    fundamentals = fundamentals.sort_values(['gvkey', 'datadate'])
    fundamentals['profit_growth_yoy'] = fundamentals.groupby('gvkey')['ebit'].pct_change(4)  # 4 quarters

    # Link to bonds via 6-digit CUSIP
    fundamentals['issuer_cusip'] = fundamentals['cusip'].str[:6]

# Trading data - calculate rolling windows
if not trading.empty:
    trading = trading.sort_values(['cusip', 'date'])

    # 5-day rolling features (Note: monthly data, so approximate)
    trading['volume_5d'] = trading.groupby('cusip')['t_volume'].rolling(1).sum().reset_index(0, drop=True)
    trading['dvolume_5d'] = trading.groupby('cusip')['t_dvolume'].rolling(1).sum().reset_index(0, drop=True)

    # 20-day rolling features
    trading['avg_volume_20d'] = trading.groupby('cusip')['t_volume'].rolling(1).mean().reset_index(0, drop=True)

# Macro data - calculate yield curve slope
if not macro_data.empty:
    macro_data['YIELD_CURVE_SLOPE'] = macro_data['DGS10'] - macro_data['DGS2']

# ============================================
# 13. SAVE FINAL PROCESSED DATASETS
# ============================================
print("\nSaving final processed datasets (with calculated features)...")
if not bond_chars.empty:
    bond_chars.to_csv('data/bond_characteristics.csv', index=False)
    print("  Saved bond_characteristics.csv")
if not ratings.empty:
    ratings.to_csv('data/credit_ratings.csv', index=False)
    print("  Saved credit_ratings.csv")
if not fundamentals.empty:
    fundamentals.to_csv('data/compustat_fundamentals.csv', index=False)
    print("  Saved compustat_fundamentals.csv")
if not trading.empty:
    trading.to_csv('data/trace_trading.csv', index=False)
    print("  Saved trace_trading.csv")
if not trace_monthly_extra.empty:
    trace_monthly_extra.to_csv('data/trace_monthly_extra.csv', index=False)
    print("  Saved trace_monthly_extra.csv")
if not stock_returns.empty:
    stock_returns.to_csv('data/crsp_returns.csv', index=False)
    print("  Saved crsp_returns.csv")
if not cds.empty:
    cds.to_csv('data/markit_cds.csv', index=False)
    print("  Saved markit_cds.csv")
if not macro_data.empty:
    macro_data.to_csv('data/fred_macro.csv', index=False)
    print("  Saved fred_macro.csv")

print("\n" + "="*50)
print("DATA EXTRACTION COMPLETE!")
print("="*50)
print("\nFinal processed files (with calculated features):")
print("  - bond_characteristics.csv")
print("  - credit_ratings.csv")
print("  - compustat_fundamentals.csv")
print("  - trace_trading.csv (monthly aggregated from WRDS)")
print("  - trace_monthly_extra.csv (monthly aggregated from raw TRACE)")
print("  - crsp_returns.csv")
print("  - markit_cds.csv (if available)")
print("  - fred_macro.csv")
print("\nRaw data files (for incremental saving):")
print("  - bond_characteristics_raw.csv")
print("  - credit_ratings_raw.csv")
print("  - compustat_fundamentals_raw.csv")
print("  - trace_trading_raw.csv")
print("  - trace_monthly_extra_raw.csv")
print("  - crsp_returns_raw.csv")
print("  - fred_macro_raw.csv")

In [ ]:
"""
INCREMENTAL EXTRACTION: SUBORDINATED BOND FEATURES
For: Bond Liquidity ML Project
Run on: Local Machine with WRDS access

This script:
1. Loads bond_characteristics_final.csv
2. Identifies subordinated bonds (SUB, JUNS, JUN)
3. Extracts missing TRACE-derived features ONLY for those bonds
4. Saves incremental files that can be appended to existing data
"""

import wrds
import pandas as pd
import numpy as np
from datetime import datetime
import os

# ============================================
# CONFIGURATION
# ============================================
START_DATE = '2015-01-01'
END_DATE = '2024-12-31'

print("="*70)
print("INCREMENTAL EXTRACTION: SUBORDINATED BOND FEATURES")
print("="*70)

# ============================================
# STEP 1: LOAD BOND CHARACTERISTICS
# ============================================
print("\n1. Loading bond_characteristics_final.csv...")

try:
    bond_chars = pd.read_csv('data/bond_characteristics_final.csv')
    print(f"   ✅ Loaded: {len(bond_chars):,} bonds")
except FileNotFoundError:
    print("   ❌ Error: bond_characteristics_final.csv not found in data/ directory")
    print("   Please ensure the file exists in: ./data/bond_characteristics_final.csv")
    exit(1)

# ============================================
# STEP 2: IDENTIFY SUBORDINATED BONDS
# ============================================
print("\n2. Identifying subordinated bonds...")

if 'security_level' not in bond_chars.columns:
    print("   ❌ Error: 'security_level' column not found")
    exit(1)

# Identify non-senior bonds (everything except SEN)
sub_mask = bond_chars['security_level'] != 'SEN'
sub_bonds = bond_chars[sub_mask].copy()

print(f"   Total bonds: {len(bond_chars):,}")
print(f"   Non-senior bonds: {len(sub_bonds):,}")
print(f"   Senior bonds: {len(bond_chars) - len(sub_bonds):,}")

if len(sub_bonds) == 0:
    print("\n   ⚠️  No subordinated bonds found. Exiting.")
    exit(0)

# Get CUSIPs
sub_cusips = sub_bonds['cusip'].dropna().unique().tolist()
print(f"\n   Unique subordinated CUSIPs to extract: {len(sub_cusips):,}")

print("\n   Security Level Distribution:")
print(sub_bonds['security_level'].value_counts())

# ============================================
# STEP 3: CONNECT TO WRDS
# ============================================
print("\n3. Connecting to WRDS...")

try:
    db = wrds.Connection()
    print("   ✅ Connected successfully")
except Exception as e:
    print(f"   ❌ Connection failed: {e}")
    exit(1)

# ============================================
# HELPER FUNCTION
# ============================================
def query_with_retry(db_conn, query, description, max_retries=3):
    """Execute query with retry logic"""
    for attempt in range(max_retries):
        try:
            print(f"      Attempt {attempt + 1}/{max_retries}: {description}")
            result = db_conn.raw_sql(query)
            print(f"      ✓ Success: {len(result):,} rows")
            return result
        except Exception as e:
            print(f"      ✗ Failed: {e}")
            if attempt < max_retries - 1:
                print("      Retrying...")
            else:
                raise
    return pd.DataFrame()

# ============================================
# STEP 4: EXTRACT TRACE MONTHLY AGGREGATED DATA
# ============================================
print("\n4. Extracting TRACE monthly aggregated data for subordinated bonds...")
print("   Note: This provides data for features 29, 32, 33, 34")
print("   (Days since last trade, Trade count, Avg trade size, Bid-ask spread)")

trace_monthly_list = []
years = range(2015, 2025)

# Process in batches to avoid query size limits
batch_size = 100
total_batches = (len(sub_cusips) + batch_size - 1) // batch_size

print(f"\n   Processing {len(sub_cusips)} CUSIPs in {total_batches} batches...")

for batch_idx in range(total_batches):
    start_idx = batch_idx * batch_size
    end_idx = min(start_idx + batch_size, len(sub_cusips))
    batch_cusips = sub_cusips[start_idx:end_idx]

    print(f"\n   Batch {batch_idx + 1}/{total_batches}: CUSIPs {start_idx + 1}-{end_idx}")

    for year in years:
        year_start = f"{year}-01-01"
        year_end = f"{year}-12-31"

        cusip_list = "','".join(batch_cusips)

        query = f"""
            SELECT
                cusip_id as cusip,
                DATE_TRUNC('month', trd_exctn_dt) as month,
                COUNT(*) as num_trades,
                SUM(entrd_vol_qt) as total_volume,
                AVG(rptd_pr) as avg_price,
                STDDEV(rptd_pr) as price_volatility,
                AVG(entrd_vol_qt) as avg_trade_size,
                MAX(trd_exctn_dt) as last_trade_date,
                COUNT(DISTINCT trd_exctn_dt) as trading_days
            FROM trace_enhanced.trace_enhanced
            WHERE cusip_id IN ('{cusip_list}')
                AND trd_exctn_dt >= '{year_start}'
                AND trd_exctn_dt <= '{year_end}'
                AND trc_st = 'T'
            GROUP BY cusip_id, DATE_TRUNC('month', trd_exctn_dt)
            ORDER BY cusip_id, month
        """

        result = query_with_retry(
            db, query,
            f"Batch {batch_idx+1}/{total_batches}, Year {year}"
        )

        if not result.empty:
            trace_monthly_list.append(result)

# Combine results
if trace_monthly_list:
    trace_monthly = pd.concat(trace_monthly_list, ignore_index=True)
    print(f"\n   ✅ Extracted {len(trace_monthly):,} bond-month observations")
else:
    trace_monthly = pd.DataFrame()
    print(f"\n   ⚠️  No TRACE data found for subordinated bonds")

# ============================================
# STEP 5: CALCULATE DERIVED FEATURES
# ============================================
if not trace_monthly.empty:
    print("\n5. Calculating derived features from TRACE data...")

    # Convert dates (with utc=True to handle timezones)
    trace_monthly['month'] = pd.to_datetime(trace_monthly['month'], utc=True).dt.tz_localize(None)
    trace_monthly['last_trade_date'] = pd.to_datetime(trace_monthly['last_trade_date'], utc=True).dt.tz_localize(None)

    # Sort by cusip and month
    trace_monthly = trace_monthly.sort_values(['cusip', 'month'])

    # Feature 29: Days since last trade (within month)
    # Calculate as days from end of month to last trade in that month
    trace_monthly['month_end'] = trace_monthly['month'] + pd.offsets.MonthEnd(0)
    trace_monthly['days_since_last_trade'] = (
        trace_monthly['month_end'] - trace_monthly['last_trade_date']
    ).dt.days

    # Feature 32: Number of trades (already have as num_trades)

    # Feature 33: Average trade size (already have as avg_trade_size)

    # Feature 34: Bid-ask spread approximation
    # Using price volatility as proxy (proper bid-ask requires more detailed TRACE data)
    trace_monthly['bid_ask_spread_proxy'] = trace_monthly['price_volatility']

    # Calculate rolling 20-day features (approximate with monthly data)
    trace_monthly = trace_monthly.sort_values(['cusip', 'month'])

    # 20-day trade count (approximate as current month trades)
    trace_monthly['trades_20d'] = trace_monthly['num_trades']

    # 20-day average trade size
    trace_monthly['avg_trade_size_20d'] = trace_monthly['avg_trade_size']

    # 20-day bid-ask spread
    trace_monthly['bid_ask_spread_20d'] = trace_monthly['bid_ask_spread_proxy']

    print(f"   ✅ Calculated derived features")
    print(f"\n   Feature Summary:")
    print(f"      days_since_last_trade: mean={trace_monthly['days_since_last_trade'].mean():.1f} days")
    print(f"      trades_20d: mean={trace_monthly['trades_20d'].mean():.1f}")
    print(f"      avg_trade_size_20d: mean={trace_monthly['avg_trade_size_20d'].mean():.1f}")
    print(f"      bid_ask_spread_20d: mean={trace_monthly['bid_ask_spread_20d'].mean():.4f}")

# ============================================
# STEP 6: EXTRACT FROM WRDS BOND RETURNS
# ============================================
print("\n6. Extracting from WRDS Bond Returns (for lagged spread, volume)...")
print("   Note: This provides features 24-28")

bondret_list = []

for year in years:
    year_start = f"{year}-01-01"
    year_end = f"{year}-12-31"

    # Convert list to SQL format
    cusip_list = "','".join(sub_cusips)

    query = f"""
        SELECT
            cusip,
            date,
            t_volume,
            t_dvolume,
            t_spread,
            t_yld_pt as ytm,
            price_eom,
            ret_eom
        FROM wrdsapps.bondret
        WHERE cusip IN ('{cusip_list}')
            AND date >= '{year_start}'
            AND date <= '{year_end}'
    """

    result = query_with_retry(db, query, f"Bond Returns {year}")
    if not result.empty:
        bondret_list.append(result)

if bondret_list:
    bondret = pd.concat(bondret_list, ignore_index=True)
    print(f"\n   ✅ Extracted {len(bondret):,} bond-month observations from bondret")

    # Convert date (with utc=True to handle timezones)
    bondret['date'] = pd.to_datetime(bondret['date'], utc=True).dt.tz_localize(None)
    bondret = bondret.sort_values(['cusip', 'date'])

    # Calculate derived features
    print("\n   Calculating bondret-derived features...")

    # Feature 26: Lagged spread
    bondret['lagged_spread'] = bondret.groupby('cusip')['t_spread'].shift(1)

    # Feature 25: 20-day average volume (use monthly volume as proxy)
    bondret['avg_volume_20d'] = bondret['t_volume']

    # Feature 27: Average monthly spread (use t_spread directly)
    bondret['avg_monthly_spread'] = bondret['t_spread']

    # Feature 28: Spread volatility (rolling)
    bondret['spread_volatility_20d'] = bondret.groupby('cusip')['t_spread'].rolling(
        window=3, min_periods=1
    ).std().reset_index(0, drop=True)

    print(f"   ✅ Calculated bondret-derived features")
else:
    bondret = pd.DataFrame()
    print(f"\n   ⚠️  No bond return data found")

# ============================================
# STEP 7: SAVE INCREMENTAL FILES
# ============================================
print("\n7. Saving incremental files...")

# Create output directory

# Save TRACE monthly features
if not trace_monthly.empty:
    output_files = [
        ('days_since_last_trade', ['cusip', 'month', 'days_since_last_trade']),
        ('trades_20d', ['cusip', 'month', 'trades_20d', 'avg_trade_size_20d']),
        ('bid_ask_spread_20d', ['cusip', 'month', 'bid_ask_spread_20d']),
    ]

    for filename, columns in output_files:
        filepath = f'data/subordinated_incremental_{filename}.csv'
        subset = trace_monthly[columns].copy()
        subset.to_csv(filepath, index=False)
        print(f"   ✅ Saved: {filepath} ({len(subset):,} rows)")

# Save bondret features
if not bondret.empty:
    # Feature 26: Lagged spread
    lagged = bondret[['cusip', 'date', 'lagged_spread']].copy()
    lagged.to_csv('data/subordinated_incremental_lagged_spread.csv', index=False)
    print(f"   ✅ Saved: lagged_spread.csv ({len(lagged):,} rows)")

    # Features 24, 25, 27, 28
    features = bondret[[
        'cusip', 'date', 'ytm', 'avg_volume_20d',
        'avg_monthly_spread', 'spread_volatility_20d'
    ]].copy()
    features.to_csv('data/subordinated_incremental_extracted_features_bondret.csv', index=False)
    print(f"   ✅ Saved: extracted_features_bondret.csv ({len(features):,} rows)")

# ============================================
# STEP 8: CLOSE CONNECTION
# ============================================
db.close()
print("\n8. WRDS connection closed")

# ============================================
# SUMMARY
# ============================================
print("\n" + "="*70)
print("EXTRACTION COMPLETE!")
print("="*70)

print(f"""
Summary:
  • Subordinated bonds processed: {len(sub_cusips):,}
  • TRACE monthly observations: {len(trace_monthly):,} bond-months
  • Bond return observations: {len(bondret):,} bond-months

Files created in data/subordinated_incremental/:
  1. days_since_last_trade.csv
  2. trades_20d.csv
  3. bid_ask_spread_20d.csv
  4. lagged_spread.csv
  5. extracted_features_bondret.csv

Next steps:
  1. Append these files to your existing feature files
  2. Re-run your merging/feature engineering pipeline
  3. Verify 100% coverage for subordinated bonds
""")

print("="*70)

In [ ]:
import pandas as pd
import numpy as np
import wrds
import os
from datetime import datetime

# ============================================
# CONFIGURATION
# ============================================
DATA_DIR = '/Users/jasongu/Desktop/data'
BONDS_FILE = os.path.join(DATA_DIR, 'bond_characteristics.csv')
OUTPUT_DAILY = os.path.join(DATA_DIR, 'trace_daily_data.csv')
OUTPUT_FEATURES = os.path.join(DATA_DIR, 'features_20_31_32_33.csv')

print("="*60)
print("EXTRACTING DAILY TRACE DATA + FEATURES 20, 31, 32, 33")
print("="*60)

start_time = datetime.now()

# ============================================
# 1. LOAD BOND LIST
# ============================================
print("\n[1/5] Loading bond list...")

bonds = pd.read_csv(BONDS_FILE)
cusips = bonds['cusip'].unique().tolist()

print(f"  Total bonds: {len(cusips):,}")

# ============================================
# 2. CONNECT TO WRDS
# ============================================
print("\n[2/5] Connecting to WRDS...")

try:
    db = wrds.Connection()
    print("  ✓ Connected")
except Exception as e:
    print(f"  ✗ Connection failed: {e}")
    exit(1)

# ============================================
# 3. EXTRACT DAILY TRACE DATA
# ============================================
print("\n[3/5] Extracting daily TRACE data...")
print("  This includes: trades, volume, prices, spreads\n")

batch_size = 100
daily_data_list = []
total_batches = (len(cusips) + batch_size - 1) // batch_size

for i in range(0, len(cusips), batch_size):
    batch = cusips[i:i+batch_size]
    batch_num = i // batch_size + 1

    cusip_list = "','".join(batch)

    # Query daily aggregates from TRACE Enhanced
    # Get everything we need for features 20, 31, 32, 33
    query = f"""
        SELECT
            cusip_id,
            trd_exctn_dt as date,
            COUNT(*) as trade_count,
            SUM(entrd_vol_qt) as daily_volume,
            AVG(entrd_vol_qt) as avg_trade_size,
            AVG(rptd_pr) as avg_price,
            STDDEV(rptd_pr) as price_std,
            MIN(rptd_pr) as min_price,
            MAX(rptd_pr) as max_price
        FROM trace_enhanced.trace_enhanced
        WHERE cusip_id IN ('{cusip_list}')
            AND trd_exctn_dt BETWEEN '2015-01-01' AND '2024-12-31'
            AND entrd_vol_qt > 0
            AND rptd_pr > 0
        GROUP BY cusip_id, trd_exctn_dt
        ORDER BY cusip_id, trd_exctn_dt
    """

    try:
        batch_start = datetime.now()
        result = db.raw_sql(query)
        batch_time = (datetime.now() - batch_start).total_seconds()

        if not result.empty:
            daily_data_list.append(result)
            print(f"  Batch {batch_num}/{total_batches}: {len(result):,} daily obs ({batch_time:.1f}s)")
        else:
            print(f"  Batch {batch_num}/{total_batches}: No data ({batch_time:.1f}s)")

        # Time estimate
        if batch_num == 5:
            avg_time = (datetime.now() - start_time).total_seconds() / 5
            estimated_total = avg_time * total_batches / 60
            print(f"\n  📊 Estimated total time: {estimated_total:.1f} minutes\n")

    except Exception as e:
        print(f"  Batch {batch_num}/{total_batches}: ERROR - {e}")

db.close()
print("\n  ✓ WRDS connection closed")

if not daily_data_list:
    print("  ✗ No data retrieved!")
    exit(1)

# ============================================
# 4. PROCESS DAILY DATA
# ============================================
print("\n[4/5] Processing daily data...")

# Combine all batches
daily_data = pd.concat(daily_data_list, ignore_index=True)
daily_data['date'] = pd.to_datetime(daily_data['date'])
daily_data = daily_data.rename(columns={'cusip_id': 'cusip'})

print(f"  Total daily observations: {len(daily_data):,}")
print(f"  Unique bonds: {daily_data['cusip'].nunique():,}")
print(f"  Date range: {daily_data['date'].min()} to {daily_data['date'].max()}")

# Sort
daily_data = daily_data.sort_values(['cusip', 'date'])

# Save daily data (optional - for reference)
daily_data.to_csv(OUTPUT_DAILY, index=False)
print(f"\n  ✓ Saved daily data: {OUTPUT_DAILY}")

# ============================================
# 5. CALCULATE FEATURES AT MONTH-END
# ============================================
print("\n[5/5] Calculating features at month-end...")

# Create month-end flags
daily_data['year_month'] = daily_data['date'].dt.to_period('M')
daily_data['is_month_end'] = daily_data.groupby(['cusip', 'year_month'])['date'].transform('max') == daily_data['date']

print(f"\n  Creating rolling window features...")

# Sort by CUSIP and date
daily_data = daily_data.sort_values(['cusip', 'date'])

# Feature 20: Days since last trade
print("    Feature 20: Days since last trade...")

def calc_days_since(group):
    days_since = []
    days_counter = 0

    for idx, row in group.iterrows():
        if row['trade_count'] > 0:
            days_counter = 0
        days_since.append(days_counter)
        days_counter += 1

    group['days_since_trade'] = days_since
    return group

daily_data = daily_data.groupby('cusip', group_keys=False).apply(calc_days_since)

# Feature 31: Number of trades in past 20 days
print("    Feature 31: Trades in past 20 days...")

daily_data['trades_20d'] = (
    daily_data.groupby('cusip')['trade_count']
    .rolling(window=20, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

# Feature 32: Average trade size in past 20 days
print("    Feature 32: Avg trade size in past 20 days...")

daily_data['avg_trade_size_20d'] = (
    daily_data.groupby('cusip')['avg_trade_size']
    .rolling(window=20, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

# Feature 33: Bid-ask spread (approximate from daily price range)
print("    Feature 33: Bid-ask spread (from price range)...")

# Approximate bid-ask spread as % of midpoint
# Using daily high-low range as proxy for bid-ask spread
daily_data['price_range'] = daily_data['max_price'] - daily_data['min_price']
daily_data['bid_ask_spread_approx'] = daily_data['price_range'] / daily_data['avg_price']

# Also calculate 20-day average spread
daily_data['bid_ask_spread_20d'] = (
    daily_data.groupby('cusip')['bid_ask_spread_approx']
    .rolling(window=20, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

# ============================================
# 6. EXTRACT MONTH-END OBSERVATIONS
# ============================================
print("\n  Filtering to month-end observations...")

# Keep only month-end observations
monthly_features = daily_data[daily_data['is_month_end']].copy()

# Select final columns
output_columns = [
    'cusip',
    'date',
    'days_since_trade',           # Feature 20
    'trades_20d',                  # Feature 31
    'avg_trade_size_20d',          # Feature 32
    'bid_ask_spread_20d'           # Feature 33 (20-day avg spread)
]

output = monthly_features[output_columns].copy()

# ============================================
# 7. SAVE
# ============================================
output.to_csv(OUTPUT_FEATURES, index=False)

print(f"\n✓ Saved: {OUTPUT_FEATURES}")

# ============================================
# SUMMARY
# ============================================
elapsed = (datetime.now() - start_time).total_seconds() / 60

print("\n" + "="*60)
print("SUMMARY")
print("="*60)

print(f"\n⏱️  Total time: {elapsed:.1f} minutes")

print(f"\nOutput (month-end observations):")
print(f"  Total observations: {len(output):,}")
print(f"  Unique bonds: {output['cusip'].nunique():,}")

print("\nFeature Coverage:")

for i, col in enumerate([
    ('days_since_trade', 'Feature 20 (Days Since Trade)'),
    ('trades_20d', 'Feature 31 (Trades in Past 20 Days)'),
    ('avg_trade_size_20d', 'Feature 32 (Avg Trade Size 20d)'),
    ('bid_ask_spread_20d', 'Feature 33 (Bid-Ask Spread 20d)')
], 20):

    feature_col, feature_name = col
    coverage = output[feature_col].notna().sum()
    pct = 100 * coverage / len(output)

    print(f"\n{feature_name}:")
    print(f"  Coverage: {coverage:,} ({pct:.1f}%)")

    if coverage > 0:
        print(f"  Mean: {output[feature_col].mean():.4f}")
        print(f"  Median: {output[feature_col].median():.4f}")
        print(f"  Std: {output[feature_col].std():.4f}")

print("\n" + "="*60)
print("NOTES")
print("="*60)
print("""
Feature 33 (Bid-Ask Spread):
  - Calculated as daily (high - low) / midpoint
  - Then averaged over 20 days
  - This is an APPROXIMATION of true bid-ask spread

  True bid-ask spread requires knowing actual bids/asks, which
  aren't available in TRACE Enhanced (only executed trades).

  The high-low range proxy is standard when true spreads unavailable
  (Corwin & Schultz, 2012; Roll, 1984).

Alternative for Feature 33:
  If you have access to actual bid-ask quotes from another source,
  use those instead. Otherwise, this approximation is acceptable.
""")

print("\n✓ Complete!")

In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================
# CONFIGURATION
# ============================================
DATA_DIR = '/Users/jasongu/Desktop/data'
TRADING_FILE = os.path.join(DATA_DIR, 'trace_trading.csv')
EXTRA_FILE = os.path.join(DATA_DIR, 'trace_monthly_extra.csv')
OUTPUT_FILE = os.path.join(DATA_DIR, 'trace_features_13_20.csv')

print("="*60)
print("CREATING TRACE FEATURES 13-20 (Monthly Approximations)")
print("="*60)

# ============================================
# 1. LOAD DATA
# ============================================
print("\n[1/4] Loading data...")

trading = pd.read_csv(TRADING_FILE)
trading['date'] = pd.to_datetime(trading['date'])

if os.path.exists(EXTRA_FILE):
    extra = pd.read_csv(EXTRA_FILE)
    extra['month'] = pd.to_datetime(extra['month'])
    print(f"  trace_trading: {len(trading):,}")
    print(f"  trace_monthly_extra: {len(extra):,}")
    use_extra = True
else:
    print(f"  trace_trading: {len(trading):,}")
    print(f"  ⚠️  trace_monthly_extra not found, using trace_trading only")
    use_extra = False

# ============================================
# 2. CREATE FEATURES FROM TRACE_TRADING
# ============================================
print("\n[2/4] Creating features from trace_trading.csv...")

# Sort by bond and date
trading = trading.sort_values(['cusip', 'date'])

# Feature 13: Volume (monthly total)
trading['volume_monthly'] = trading['t_dvolume']

# Feature 17: Lagged spread (t-1)
trading['lagged_spread'] = trading.groupby('cusip')['t_spread'].shift(1)

# Feature 18: Avg spread (monthly) - this is just t_spread for monthly data
trading['avg_spread_monthly'] = trading['t_spread']

# Feature 19: Spread volatility (rolling 3-month std)
trading['spread_volatility_3m'] = (
    trading.groupby('cusip')['t_spread']
    .rolling(window=3, min_periods=2)
    .std()
    .reset_index(level=0, drop=True)
)

# Feature 20: Months since last trade (FIXED VERSION)
print("  Calculating months since last trade...")

# Create a helper column for cumulative count
trading['has_volume'] = trading['t_volume'].notna().astype(int)

# Method: Use cumsum to track periods between trades
def calculate_months_since_trade(group):
    """Calculate months since last trade for a single bond"""
    # Create cumulative sum that resets at each trade
    trade_groups = group['has_volume'].cumsum()

    # For each trade group, calculate months since that trade started
    months_since = group.groupby(trade_groups).cumcount()

    # Set to 0 for months with actual trades
    months_since = np.where(group['has_volume'] == 1, 0, months_since)

    # Set to NaN if never traded before
    first_trade = (group['has_volume'].cumsum() == 0)
    months_since = np.where(first_trade, np.nan, months_since)

    return months_since

trading['months_since_trade'] = (
    trading.groupby('cusip', group_keys=False)
    .apply(lambda x: pd.Series(calculate_months_since_trade(x), index=x.index))
)

print(f"  ✓ Created basic features")

# ============================================
# 3. MERGE WITH TRACE_MONTHLY_EXTRA
# ============================================
if use_extra:
    print("\n[3/4] Merging with trace_monthly_extra...")

    # Rename month to date for merging
    extra_renamed = extra.rename(columns={'month': 'date', 'cusip_id': 'cusip'})

    # Ensure date is datetime type (handle timezone-aware datetimes)
    extra_renamed['date'] = pd.to_datetime(extra_renamed['date'], utc=True).dt.tz_localize(None)

    # Ensure cusip is same type
    if extra_renamed['cusip'].dtype != trading['cusip'].dtype:
        extra_renamed['cusip'] = extra_renamed['cusip'].astype(str)
        trading['cusip'] = trading['cusip'].astype(str)

    # Merge
    merged = trading.merge(
        extra_renamed[['cusip', 'date', 'trade_count', 'avg_trade_size',
                      'dealer_trades', 'customer_trades']],
        on=['cusip', 'date'],
        how='left'
    )

    # Update features with extra data
    merged['trades_monthly'] = merged['trade_count']
    merged['avg_trade_size_monthly'] = merged['avg_trade_size']

    # Feature 16: Dealer activity (approximation of unique dealers)
    merged['dealer_activity'] = merged['dealer_trades']

    coverage_trades = merged['trades_monthly'].notna().sum()
    coverage_size = merged['avg_trade_size_monthly'].notna().sum()

    print(f"  ✓ Merged with extra data")
    print(f"    Trades coverage: {coverage_trades:,} ({100*coverage_trades/len(merged):.1f}%)")
    print(f"    Trade size coverage: {coverage_size:,} ({100*coverage_size/len(merged):.1f}%)")

else:
    print("\n[3/4] Skipping extra merge (file not found)...")
    merged = trading.copy()

    # Approximate features from available data
    # If t_volume exists, assume at least 1 trade
    merged['trades_monthly'] = np.where(merged['t_volume'].notna(), 1, np.nan)

    # Approximate avg trade size as total volume / 1 (rough estimate)
    merged['avg_trade_size_monthly'] = merged['t_volume']

    # No dealer info
    merged['dealer_activity'] = np.nan

    print(f"  ⚠️  Using approximations (no trace_monthly_extra)")

# ============================================
# 4. SAVE OUTPUT
# ============================================
print("\n[4/4] Saving features...")

output_columns = [
    'cusip',
    'date',
    'volume_monthly',              # Feature 13
    'trades_monthly',              # Feature 14
    'avg_trade_size_monthly',      # Feature 15
    'dealer_activity',             # Feature 16 (approximation)
    'lagged_spread',               # Feature 17
    'avg_spread_monthly',          # Feature 18
    'spread_volatility_3m',        # Feature 19
    'months_since_trade'           # Feature 20
]

existing_cols = [col for col in output_columns if col in merged.columns]
output_data = merged[existing_cols].copy()

output_data.to_csv(OUTPUT_FILE, index=False)

print(f"\n✓ Saved: {OUTPUT_FILE}")

# ============================================
# SUMMARY
# ============================================
print("\n" + "="*60)
print("SUMMARY")
print("="*60)

print(f"\nTotal observations: {len(output_data):,}")
print(f"Unique bonds: {output_data['cusip'].nunique():,}")

feature_names = {
    'volume_monthly': 'Feature 13 (Volume)',
    'trades_monthly': 'Feature 14 (Trades)',
    'avg_trade_size_monthly': 'Feature 15 (Avg Trade Size)',
    'dealer_activity': 'Feature 16 (Dealer Activity)',
    'lagged_spread': 'Feature 17 (Lagged Spread)',
    'avg_spread_monthly': 'Feature 18 (Avg Spread)',
    'spread_volatility_3m': 'Feature 19 (Spread Volatility 3m)',
    'months_since_trade': 'Feature 20 (Months Since Trade)'
}

usable_count = 0

for col, name in feature_names.items():
    if col in output_data.columns:
        coverage = output_data[col].notna().sum()
        pct = 100 * coverage / len(output_data)

        print(f"\n{name}:")
        print(f"  Coverage: {coverage:,} ({pct:.1f}%)")

        if coverage > 0:
            print(f"  Mean: {output_data[col].mean():.4f}")
            print(f"  Std: {output_data[col].std():.4f}")

            if pct >= 50:
                print(f"  ✅ USABLE")
                usable_count += 1
            else:
                print(f"  ⚠️  Low coverage")

print("\n" + "="*60)
print(f"USABLE FEATURES FROM THIS FILE: {usable_count}/8")
print("="*60)

print("\n" + "="*60)
print("IMPORTANT NOTES")
print("="*60)
print("""
These features use MONTHLY approximations instead of daily rolling windows:
- Features 13-15: Monthly totals instead of 5-day/20-day windows
- Feature 16: Dealer trade count (NOT unique dealers - would need daily data)
- Feature 19: 3-month rolling std instead of 20-day
- Feature 20: Months since trade instead of days

This is standard practice when working with monthly bond data.

If trace_monthly_extra.csv was found and merged, Features 14-16 will have
better coverage. Otherwise, they use approximations from trace_trading.csv.
""")

print("\n✓ Complete!")